In [19]:
import pickle
import pandas as pd

Attached to the email, you will find a binary(pickled) file invoices_new.pkl, which contains information of invoices. You may assume that the information has been extracted by scanning paper invoices using OCR. Each invoice has its own properties (id, name, creation date etc.) and a list of invoice items. Like invoice, invoice items also have their own properties.
In the same directory, there is also a file containing a list of IDs of expired invoices - expired_invoices.txt

Write a DataExtractor class that has the functionality to load the dataset and transform the unstructured data into a flat data with the following columns:
invoice_id: int
created_on: datetime64[ns]
invoiceitem_id: int
invoiceitem_name: str
type: str (use this conversion table: {0: 'Material', 1: 'Equipment', 2: 'Service', 3: 'Other'})
unit_price: int
total_price: int (unit_price*quantity)
percentage_in_invoice: float (unit_price*quantity / invoice_total)
is_expired: bool (whether invoice_id is contained expired_invoices.txt or not)

Make sure that the columns in the resulting dataframe are of the correct type, as described above. The final dataframe must be sorted by invoice_id and invoiceitem_id in the ascending order.


In [52]:
class DataExtractor:
    def __init__(self, pickle_file, txt_file):
        self.pickle_path = pickle_file
        self.txt_path = txt_file
        self.data = self.load_data()
        
    def load_data(self):
        # load pickle part
        with open(self.pickle_path, "rb") as d:
            pickle_part = pickle.load(d)
        
        return pd.DataFrame(pickle_part)
    
        
    def load_expired_ids(self): 
        # load txt data
        # https://chatgpt.com/share/1aaea9ee-6875-450b-a186-2ca50f7108f4
        with open(self.txt_path, 'r') as file:
            numbers = [int(num.strip()) for num in file.read().split(',')]
        
        return numbers
            
    def create_is_expired_col(self):
        numbers = self.load_expired_ids()
        is_expired_list = []
        for item in self.data["id"]:
            if item in numbers:
                is_expired_list.append(True)
            else:
                is_expired_list.append(False)
        self.data["expired_list"] = is_expired_list
        
    
    def convert_datatypes(self):
        self.data["id"] = self.data["id"].replace('O', '0', regex=True)
        self.data["id"] = self.data["id"].astype(int)
        self.data.rename(columns={"id": "invoice_id"}, inplace=True)
        # https://chatgpt.com/share/abf3a93a-1045-4bdd-a286-2d44c4d939c4
        self.data["created_on"] = pd.to_datetime(self.data["created_on"], errors="coerce")
        return self.data.sort_values(by="invoice_id", ascending=True)
        

        
    
        

In [53]:
x = DataExtractor("invoices_new.pkl", "expired_invoices.txt")
x.convert_datatypes()

,invoice_id,created_on,items,expired_list
60,301695,2019-04-26,"[{'item': {'id': 166227, 'name': 'ii_166227', ...",False
27,304245,2019-03-17,"[{'item': {'id': 121446, 'name': 'ii_121446', ...",False
94,305869,2019-09-11,"[{'item': {'id': 132142, 'name': 'ii_132142', ...",True
96,306163,2019-09-18,"[{'item': {'id': 136017, 'name': 'ii_136017', ...",False
11,307175,2019-05-19,"[{'item': {'id': 134663, 'name': 'ii_134663', ...",True
...,...,...,...,...
97,3733750,2019-08-24,"[{'item': {'id': 180383, 'name': 'ii_180383', ...",False
8,3740890,2019-09-30,"[{'item': {'id': 160942, 'name': 'ii_160942', ...",False
38,3814760,2019-07-12,"[{'item': {'id': 195222, 'name': 'ii_195222', ...",False
33,3852900,2019-06-29,"[{'item': {'id': 150069, 'name': 'ii_150069', ...",False


# https://chatgpt.com/share/ba0d9f5f-193b-481c-a481-67dc780df49c

here is the apporach of dealing with the items column, but I did not manage to do that.